In [ ]:
import os
import json
import seaborn as sns
import matplotlib.pyplot as plt 
import numpy as np
from sklearn import metrics
from sklearn.metrics import brier_score_loss
import torch
from torchmetrics.classification import BinaryCalibrationError
ece_metric = BinaryCalibrationError(n_bins=15, norm='l1')
nll_metric = torch.nn.BCELoss()

In [ ]:
def convert_to_array(str_input):
    preds = []
    runs = str_input.split('\n')
    for run in runs:
        nonpos_prob, pos_prob = run.replace("[", "").replace("]", "").split()
        nonpos_prob, pos_prob = float(nonpos_prob), float(pos_prob)
        pred = [nonpos_prob, pos_prob]
        preds.append(pred)
    return preds

In [ ]:
key_list = [
    'Accuracy',
    'Precision',
    'Recall',
    'F1 Score',
    'AUC ROC', 
    'ECE', 
    'NLL', 
    'Brier Score'
]

In [ ]:
PLM_LIST = ['esmc_600m', 'Rostlab/ProstT5', 'ElnaggarLab/ankh-large', 'facebook/esm2_t33_650M_UR50D', 'Rostlab/prot_bert']

In [ ]:
PLM_dict = {
    'esmc_600m': 'ESMC',
    'Rostlab/ProstT5': 'ProstT5',
    'ElnaggarLab/ankh-large': 'Ankh',
    'facebook/esm2_t33_650M_UR50D': 'ESM2',
    'Rostlab/prot_bert': 'Prot Bert'
}

In [ ]:
def return_metrics(plm_model_name, method, seed:int=None, blosum62:str=None):
    result_metrics = {}

    if method in ["EDL", "FEDL", "TS", "SGLD"]:
        dir_suffix = "-" + method
    elif method in [""]:
        dir_suffix = method
    else:
        dir_suffix = "-UQ-" #+ remove_digits_from_string(method)

    if blosum62 is not None:
        dir_suffix += blosum62

    # result_save_dir = f"Predict-Results-VBT{dir_suffix}-seed-{seed}"
    if dataset in ["Virus", "Bacteria", "Tumor"]:
        result_save_dir = f"Predict-Results-VBT{dir_suffix}"
    elif dataset in ["ToxDL", "SDAP2"]:
        result_save_dir = f"Predict-Results{dir_suffix}"
    
    if seed is not None:
        result_save_dir += f"-seed-{seed}"

    # pred_filename = f"pred_{method}.json"
    # # pred_filename = f"pred_{method}_only_ez.json"
    if method in ["LA32", "SVDKL32", "DVBLL32"]:
        file_suffix = "32"
    elif method in ["LA128", "SVDKL128", "DVBLL128"]:
        file_suffix = "128"
    else:
        file_suffix = ""

    if dataset == targetset:
        pred_filename = "{}_{}{}.json".format(dataset, testset, file_suffix)
    else:
        pred_filename = "data_{}_target_{}_{}{}.json".format(dataset, targetset, testset, file_suffix) 

    with open(os.path.join(result_save_dir, pred_filename), 'r') as stream:
        data_loaded = json.load(stream)

    if method in ["", "TS"]:
        y_true, y_pred, y_Sigma, names = [], [], [], []
        for protein_name in data_loaded:
            pred_prob_plms = data_loaded[protein_name]['pred_prob']
            # pred_unct_plms = data_loaded[protein_name]['pred_u']
            y_hat = float(pred_prob_plms[plm_model_name])
            # y_sigma = float(pred_unct_plms[plm_model_name].replace('[','').replace(']',''))
            y = int(data_loaded[protein_name]['true_label'])
            y_true.append(y)
            y_pred.append(y_hat)
            y_Sigma.append(0)
            names.append(protein_name)
    elif method in ["EDL", "FEDL"]:
        y_true, y_pred, y_Sigma, names = [], [], [], []
        for protein_name in data_loaded:
            pred_prob_plms = data_loaded[protein_name]['pred_prob']
            pred_unct_plms = data_loaded[protein_name]['pred_u']
            y_hat = float(pred_prob_plms[plm_model_name])
            y_sigma = float(pred_unct_plms[plm_model_name].replace('[','').replace(']',''))
            y = int(data_loaded[protein_name]['true_label'])
            y_true.append(y)
            y_pred.append(y_hat)
            y_Sigma.append(y_sigma)
            names.append(protein_name)
    else:
        y_true, y_pred, y_Sigma, names = [], [], [], []
        for protein_name in data_loaded:
            pred_prob_plms = data_loaded[protein_name]['pred_prob']
            pred_prob = convert_to_array(pred_prob_plms[plm_model_name])
            y_hat = np.array(pred_prob)[:,1].mean()
            y_sigma = np.array(pred_prob)[:,1].std()
            y = int(data_loaded[protein_name]['true_label'])
            y_true.append(y)
            y_pred.append(y_hat)
            y_Sigma.append(y_sigma)
            names.append(protein_name)

    # for protein_name, y_hat, y_sigma in zip(names, y_pred, y_Sigma):
    #     result_metrics[protein_name+"_prob"] = y_hat
    #     # result_metrics[protein_name+"_unct"] = y_sigma

    for protein_name, y_hat, y_sigma, y in zip(names, y_pred, y_Sigma, y_true):
        result_metrics[protein_name] = {}
        result_metrics[protein_name]["prob"] = y_hat
        result_metrics[protein_name]["unct"] = y_sigma
        result_metrics[protein_name]["labl"] = y

    return result_metrics, (names, y_pred, y_Sigma)

In [ ]:
def get_result_dict(method_list, blosum62:str=None):
    Metrics = {}
    # method_list = ["", "SVDKL", "LA", "TS", "DVBLL", "EDL", "SGLD", "SWAG", "DROPOUT"]

    for seed in [1, 2, 3, 4, 5]:
        for plm_model_name in PLM_LIST:
            for method in method_list:
                metric_results, (names, y_pred, y_Sigma) = return_metrics(plm_model_name, method, seed, blosum62)
                # for key in key_list:
                #     Metrics[key].append(metric_results[key])
                for key in metric_results:
                    # if key not in Metrics:
                    #     Metrics[key] = []
                    # Metrics[key].append(metric_results[key])
                    if key not in Metrics:
                        Metrics[key] = {}
                    if PLM_dict[plm_model_name] not in Metrics[key]:
                        Metrics[key][PLM_dict[plm_model_name]] = {}
                    if method not in Metrics[key][PLM_dict[plm_model_name]]:
                        Metrics[key][PLM_dict[plm_model_name]][method] = {}
                    Metrics[key][PLM_dict[plm_model_name]][method][seed] = metric_results[key]
    return Metrics

In [ ]:
testset = "test" #"test" # "independent" or "test" or "valid"

# To evaluate models on ImmunoVirus dataset for In-Distribution scenario, update this lines for Out-of-Distribution scenario
dataset = "Virus" #"Tumor" #"Bacteria" #"Virus"
targetset = "Virus" #"Tumor" #"Bacteria" #"Virus"

In [ ]:
EPSILON=1e-12
method_list = ["", "SVDKL", "LA", "TS", "DVBLL", "EDL", "FEDL", "SGLD", "SWAG", "DROPOUT"] #
result_dict = get_result_dict(method_list)

In [ ]:
def get_results(result_dict, plm_model_name, model_name, seed):

    y_pred, y_true = [], []
    if model_name == 'Ensemble':    
        # y_pred, y_true = [], []
        for protein_name in result_dict:
            y_p = []
            for plm_name in PLM_LIST:
                plm_model_name = PLM_dict[plm_name]
                y_p.append(result_dict[protein_name][plm_model_name][''][seed]['prob'])
            y_pred.append(np.mean(y_p))

            # y_pred.append(result_dict[protein_name][plm_model_name][model_name][seed]['prob'])
            y_true.append(result_dict[protein_name][plm_model_name][''][seed]['labl'])
    else:
        for protein_name in result_dict:
            y_pred.append(result_dict[protein_name][plm_model_name][model_name][seed]['prob'])
            y_true.append(result_dict[protein_name][plm_model_name][model_name][seed]['labl'])

    ece = ece_metric(torch.tensor(y_pred, dtype=torch.float), torch.tensor(y_true, dtype=torch.int)).item()
    nll = nll_metric(torch.tensor(y_pred, dtype=torch.float), torch.tensor(y_true, dtype=torch.float)).item()

    cm_matrix = metrics.confusion_matrix(y_true, np.array(np.array(y_pred)>=0.5, dtype=np.int8))
    # print(cm_matrix, plm_model_name, seed, model_name)

    precision = cm_matrix[1][1] / max(EPSILON, cm_matrix[1][1]+cm_matrix[0][1])
    recall = cm_matrix[1][1] / max(EPSILON, cm_matrix[1][1]+cm_matrix[1][0])
    f1_score = (2*precision*recall) / max(EPSILON, precision+recall)
    accuracy = (cm_matrix[0][0]+cm_matrix[1][1]) / (cm_matrix[0][0]+cm_matrix[1][1]+cm_matrix[1][0]+cm_matrix[0][1])

    # precision = metrics.precision_score(y_true, np.array(np.array(y_pred)>=0.5, dtype=np.int8))
    # recall = metrics.recall_score(y_true, np.array(np.array(y_pred)>=0.5, dtype=np.int8))
    # f1_score = (2*precision*recall) / max(EPSILON, precision+recall)
    # accuracy = (cm_matrix[0][0]+cm_matrix[1][1]) / (cm_matrix[0][0]+cm_matrix[1][1]+cm_matrix[1][0]+cm_matrix[0][1])

    fpr, tpr, thresholds = metrics.roc_curve(np.array(y_true)+1, y_pred, pos_label=2)
    auc_roc = metrics.auc(fpr, tpr)

    bsl = brier_score_loss(y_true, y_pred)

    return accuracy, precision, recall, f1_score, auc_roc, ece, nll, bsl

In [ ]:
Metrics = {}

method_list = ["", "LA", "DVBLL", "SWAG", "DROPOUT", "SVDKL", "SGLD", "EDL", "Ensemble", "TS"] 

for plm_model_name in PLM_LIST:
    plm_metrics = Metrics[PLM_dict[plm_model_name]] = {}
    for model_name in method_list:
        model_metrics = plm_metrics[model_name] = {}
        for key in key_list:
            model_metrics[key] = []

for plm_model_name in PLM_LIST:
    for model_name in method_list:
        for seed in [1, 2, 3, 4, 5]:

            accuracy, precision, recall, f1_score, auc_roc, ece, nll, bsl = get_results(result_dict, PLM_dict[plm_model_name], model_name, seed)
            Metrics[PLM_dict[plm_model_name]][model_name]['Accuracy'].append(accuracy)
            Metrics[PLM_dict[plm_model_name]][model_name]['Precision'].append(precision)
            Metrics[PLM_dict[plm_model_name]][model_name]['Recall'].append(recall)
            Metrics[PLM_dict[plm_model_name]][model_name]['F1 Score'].append(f1_score)
            Metrics[PLM_dict[plm_model_name]][model_name]['AUC ROC'].append(auc_roc)
            Metrics[PLM_dict[plm_model_name]][model_name]['ECE'].append(ece)
            Metrics[PLM_dict[plm_model_name]][model_name]['NLL'].append(nll)
            Metrics[PLM_dict[plm_model_name]][model_name]['Brier Score'].append(bsl)

In [ ]:
import pandas as pd
model_name_dict = {
    "": "Deterministic", 
    "TS": "TS", 
    "LA": "LA", "LA32": "LA32", "LA128": "LA128",  
    "DVBLL": "DVBLL", "DVBLL32": "DVBLL32", "DVBLL128": "DVBLL128",
    "EDL": "EDL", 
    "FEDL": "FEDL",
    "SWAG": "SWAG", 
    "DROPOUT": "MCD", 
    "SVDKL": "DKL", "SVDKL32": "DKL32", "SVDKL128": "DKL128",
    "SGLD": "SGLD", 
    "Ensemble": "Ensemble",
}

In [ ]:
method_list = ["", "TS", "LA", "DVBLL", "EDL", "SWAG", "DROPOUT", "SVDKL", "SGLD"]

table_metrics = {}
table_metrics['PLM'] = []
for key in key_list:
    table_metrics[key] = []

idx_dict = {}
model_idx = 0

# plm_model_name = PLM_dict[PLM_LIST[idx]] # Change idx from 0 to 4
for plm_name in PLM_LIST:
    plm_model_name = PLM_dict[plm_name]

    for model_name in method_list:
        table_metrics['PLM'].append(plm_model_name)
        idx_dict[model_idx] = model_name_dict[model_name]
        for key in key_list:
            key_results = Metrics[plm_model_name][model_name][key]
            table_metrics[key].append(f'{np.mean(key_results): 0.04f}'+ '$_{\\pm' + f'{np.std(key_results): 0.04f}' +'}$')
        model_idx += 1

table_metrics['PLM'].append('-')
idx_dict[model_idx] = 'Ensemble'
for key in key_list:
    key_results = Metrics[plm_model_name]['Ensemble'][key]
    table_metrics[key].append(f'{np.mean(key_results): 0.04f}'+ '$_{\\pm' + f'{np.std(key_results): 0.04f}' +'}$')
# model_idx += 1

In [ ]:
df = pd.DataFrame(table_metrics)
df.rename(index=idx_dict, inplace=True)
# df

In [ ]:
df

# Generate Result Figures

In [ ]:
import matplotlib.pyplot as plt

# Set a professional style for the plots
plt.style.use('ggplot')

In [ ]:
import math

def plot_method_metrics(results_dict, save_file:bool=True):
    """
    Generates a single figure containing subplots for each metric, comparing different methods.
    
    The bar height represents the mean score, and error bars show the
    minimum and maximum range observed across multiple seeds/runs.

    The plots are generated from the input dictionary, where:
    - Keys are the method names.
    - Values are dictionaries of metric names and their corresponding scores (lists of numbers).

    Args:
        results_dict (dict): A dictionary mapping method names to metric scores (list of numbers).
                             Example: {'Method A': {'Accuracy': [0.85, 0.84, 0.86], 'F1 Score': [...]}, ...}
        filename_prefix (str): Prefix for the saved plot filename (e.g., 'project_results').
    """
    if not results_dict:
        print("The results dictionary is empty. Cannot generate plots.")
        return

    # colors = ["tab:blue"] + ["tab:green"] * 6 + ["tab:red"] * 3
    
    # 1. Calculate Mean, Min, Max, and Error components for all metrics and methods
    calculated_results = {}
    
    for method_name, metrics in results_dict.items():
        for metric, scores in metrics.items():
            if metric not in calculated_results:
                calculated_results[metric] = []

            # Ensure scores are a list of numbers for NumPy processing
            scores_array = np.array(scores)
            
            if scores_array.size == 0:
                print(f"Warning: No scores found for {method_name} - {metric}. Skipping.")
                continue

            mean_val = np.mean(scores_array) #np.median(scores_array) #
            min_val = np.min(scores_array)
            max_val = np.max(scores_array)
            
            # Error bars based on Max/Min from the mean:
            # lower_err is the distance from the mean to the min value (always positive)
            lower_err = mean_val - min_val 
            # upper_err is the distance from the mean to the max value (always positive)
            upper_err = max_val - mean_val

            calculated_results[metric].append({
                'method': model_name_dict[method_name],
                'mean': mean_val,
                'min': min_val,
                'max': max_val,
                'lower_err': lower_err,
                'upper_err': upper_err
            })

    print("Data successfully processed (Mean, Min, Max calculated) and ready for plotting.")
    print("-" * 60)

    metric_names = list(calculated_results.keys())
    num_metrics = len(metric_names)

    if num_metrics == 0:
        print("No valid metrics found for plotting after processing.")
        return

    # 2. Setup subplot grid (max 2 columns for better horizontal viewing)

    # num_rows = min(num_metrics, 4)

    num_rows = min(num_metrics, 2)
    num_cols = int(math.ceil(num_metrics / num_rows))
    
    # Adjust figure size for better readability of subplots
    fig, axes = plt.subplots(num_rows, num_cols, 
                            #  figsize=(12 * num_cols, 4 * num_rows), 
                             figsize=(6 * num_cols, 4 * num_rows), 
                             squeeze=False) # squeeze=False ensures axes is always 2D
    
    # Flatten the axes array for easier iteration
    axes = axes.flatten()
    
    # Add a main title for the whole figure
    # fig.suptitle(f'Evaluative results in Immuno-{targetset} dataset', 
    #              fontsize=18, fontweight='bold', y=0.98)

    
    
    # 3. Iterate through each metric and plot onto a specific subplot
    for i, metric in enumerate(metric_names):
        data_list = calculated_results[metric]
        ax = axes[i]
        
        # Sort data by mean score (descending for easier comparison)
        # data_list.sort(key=lambda x: x['mean'], reverse=True)
        # if metric in ['ECE', 'NLL', 'Brier Score']:
        #     data_list.sort(key=lambda x: x['mean'], reverse=False)
        # else:
        #     data_list.sort(key=lambda x: x['mean'], reverse=True)
        
        methods = [d['method'] for d in data_list]
        mean_scores = [d['mean'] for d in data_list]
        min_scores = [d['min'] for d in data_list]
        max_scores = [d['max'] for d in data_list]
        
        # Prepare yerr in the required 2xN format: [[lower_errs], [upper_errs]]
        yerr_low = [d['lower_err'] for d in data_list]
        yerr_high = [d['upper_err'] for d in data_list]
        yerr = [yerr_low, yerr_high]

        # Use a contrasting color map for the bars
        colors = plt.cm.cividis(np.linspace(0, 1, len(methods)))

        # Create the bar plot on the current subplot axis
        bars = ax.bar(methods, mean_scores, 
                      yerr=yerr,             # Add custom error bars (Min/Max range)
                      capsize=5,             # Cap width for error bars
                      color=colors, 
                      alpha=0.8, 
                      edgecolor='black', 
                      linewidth=0.5)

        # Customizing the subplot
        # ax.set_title(metric, fontsize=28, fontweight='medium')
        if metric in ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC ROC']:
            ax.set_ylabel(metric + '(' + u'\u2191' + ')', fontsize=28)
        elif metric in ['ECE', 'NLL', 'Brier Score']:
            ax.set_ylabel(metric + '(' + u'\u2193' + ')', fontsize=28)
        else:
            ax.set_ylabel(metric, fontsize=28)
        # ax.set_yticks(fontsize=16)
        ax.tick_params(axis='y', labelsize=20)
        # ax.set_xlabel(metric, fontsize=28)
        ax.grid(axis='y', linestyle=':', alpha=0.6)
        
        # Add data labels on top of the error bars (showing only the Mean for cleaner look)
        for j, bar in enumerate(bars):
            mean_val = mean_scores[j]
            max_val = max_scores[j]
            
            label = f'{mean_val:.3f}' 
            
            # Position the text slightly above the max score (top of the error bar)
            text_y_position = max_val + (ax.get_ylim()[1] * 0.01) # Small padding
            
            ax.text(bar.get_x() + bar.get_width()/2.0, text_y_position, label,
                    ha='center', va='bottom', fontsize=18) 

        # Adjust y-axis limits
        max_score_overall = max(max_scores) if max_scores else 0
        min_score_overall = min(min_scores) if min_scores else 0
        # if max_score_overall > 0:
        #     ax.set_ylim(0, max_score_overall * 1.15)

        ax.set_ylim(min_score_overall * 0.925, max_score_overall * 1.075)
        
        # Rotate x-axis labels
        ax.set_xticks(np.arange(len(methods)))
        ax.set_xticklabels(methods, rotation=20, ha='right', fontsize=20)
        # plt.xticks(rotation=20, ha='right', fontsize=14)
        
    # Hide any unused subplots
    for k in range(num_metrics, num_rows * num_cols):
        fig.delaxes(axes[k])
        
    # Final layout adjustments and save
    plt.tight_layout(rect=[0, 0, 1, 0.95]) # Adjust rect to make space for suptitle
    
    if targetset == dataset:
        plt.suptitle(f"Comparative Performance on Immuno{targetset} dataset", fontsize=28)
    else:
        plt.suptitle(f"Comparative Performance on Immuno{targetset} dataset of models trained with Immuno{dataset} dataset", fontsize=28)

    if save_file:
        plot_filename = f"{dataset}_to_{targetset}_all_metrics_comparison.eps"
        plt.savefig(plot_filename, bbox_inches='tight')
        print(f"\nSingle figure with all subplots saved successfully to: {plot_filename}")

        # plot_filename = f"{dataset}_to_{targetset}_all_metrics_comparison.svg"
        # plt.savefig(plot_filename, bbox_inches='tight')

    # Note: To display the plots immediately, uncomment the line below:
    plt.show()

In [ ]:
plot_method_metrics(Metrics['ESMC'], save_file=True) # Change 'ESMC' to other PLM names as needed